# Notebook 03 — QC Output Review

**Role in the pipeline:** purely *optional, read-only inspection* of a
QC output folder produced by Notebook 02 (or any other stage that follows
the same layout). This notebook produces nothing downstream consumes —
it is for a human eyeball.

```text
Notebook 02 (or any stage)
   └── ...__qc_outputs/
            sheet_<x>/{data,qc,figures,reports,tables}
            logs/manifest.json
                        │
                        ▼
                03_qc_output_review.ipynb   ◄── THIS NOTEBOOK
                  (inventory + CSV preview + figure display)
```

**What it does**

1. Recursively scans `OUTPUT_ROOT` and builds a tidy file inventory.
2. Optionally filters by a substring keyword (e.g. `"phstd"`, `"sheet_0"`,
   `"ta"`).
3. Previews the first N rows of every matching CSV.
4. Displays every matching image (JPEG/PNG) inline.

Because there are no downstream consumers, **no `manifest.json` is
written.** The "Ten Simple Rules" guidance on recording provenance applies
to outputs that feed something else; an interactive viewer has nothing
to record. See `03_qc_output_review.README.md` for the rationale.


## Parameters

Single tagged `parameters` cell. The default `OUTPUT_ROOT` matches the
folder Notebook 02 produces by default (`<workbook>__qc_outputs`); change
it to look at any other stage's output root.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# Folder to scan. Default = Notebook 02's default output root.
OUTPUT_ROOT = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_prelim_data__qc_outputs"

# Optional substring keyword (case-insensitive). Matched against both
# filename and the path relative to OUTPUT_ROOT, so "sheet_0" catches every
# file under that sheet's folder.
# Examples:  None  |  "phstd"  |  "ta"  |  "sheet_0"  |  "report"
KEYWORD_FILTER = None

# How many rows to preview from each CSV.
CSV_PREVIEW_ROWS = 10

# Display toggles.
SHOW_ALL_CSV_TABLES = True
SHOW_ALL_FIGURES = True

# Optional: inspect just one file (full path). Leave both as None to use
# the bulk preview above.
SINGLE_CSV_TO_PREVIEW = None
SINGLE_IMAGE_TO_PREVIEW = None


## Setup

No `%pip install` here — this notebook only needs `pandas` (and
`matplotlib` if you display figures). Inspection helpers are in
`oa_inspect.py`.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from oa_common import die
from oa_inspect import (
    filter_inventory,
    get_csv_files,
    get_image_files,
    list_output_files,
    preview_csv_table,
    show_image,
)


def display_dataframe(df: pd.DataFrame, title: str | None = None) -> None:
    """Show a DataFrame nicely in a notebook, or print as a fallback."""
    if title:
        print(title)
    if display is not None:
        display(df)
    else:
        print(df.to_string(index=False))


## Build inventory

Recursively scan `OUTPUT_ROOT` and apply the keyword filter (if any).
Fails fast with a clear message if the folder is missing.


In [ ]:
output_root = Path(OUTPUT_ROOT).expanduser().resolve()

inventory_all = list_output_files(output_root)
inventory = filter_inventory(inventory_all, KEYWORD_FILTER)

print(f"Output root   : {output_root}")
print(f"Files (total) : {len(inventory_all)}")
print(f"Files (match) : {len(inventory)}"
      + (f"   [keyword: {KEYWORD_FILTER!r}]" if KEYWORD_FILTER else ""))


## View inventory table

Sorted by `parent` then `suffix` so all the CSVs in a given folder appear
together. The display shows the relative path (readable on narrow
screens); the full path is still in the dataframe if you need to copy it.


In [ ]:
inventory_view = (
    inventory.sort_values(["parent", "suffix", "name"])
             .reset_index(drop=True)
             [["relative_path", "suffix", "size_kb", "parent", "name"]]
)
display_dataframe(inventory_view, title="\nMatched output files\n")


## Preview CSV tables

For every matching CSV, read just the first `CSV_PREVIEW_ROWS` rows
(streaming, not the whole file) and show them. Errors on individual files
are reported but do not stop the loop — useful when one CSV happens to be
malformed.


In [ ]:
csv_files = get_csv_files(inventory)
print(f"CSV tables found: {len(csv_files)}")

if SHOW_ALL_CSV_TABLES and csv_files:
    for csv_path in csv_files:
        print("\n" + "=" * 100)
        print(f"CSV preview: {csv_path.name}")
        print(f"Folder     : {csv_path.parent}")
        try:
            df_preview = preview_csv_table(csv_path, nrows=CSV_PREVIEW_ROWS)
            display_dataframe(df_preview)
            print(f"Shape previewed: {df_preview.shape}")
        except Exception as e:
            print(f"Could not read CSV: {csv_path}")
            print(f"Reason: {e}")
elif not csv_files:
    print("(no CSV files matched)")
else:
    print("CSV preview is disabled (SHOW_ALL_CSV_TABLES = False).")


## Display figures

Inline render of every matching .png/.jpg/.jpeg/.webp. Same per-file
error tolerance as the CSV section.


In [ ]:
image_files = get_image_files(inventory)
print(f"Figures found: {len(image_files)}")

if SHOW_ALL_FIGURES and image_files:
    for img_path in image_files:
        print("\n" + "=" * 100)
        print(f"Figure: {img_path.name}")
        print(f"Folder: {img_path.parent}")
        try:
            show_image(img_path, title=img_path.name)
        except Exception as e:
            print(f"Could not display image: {img_path}")
            print(f"Reason: {e}")
elif not image_files:
    print("(no figure files matched)")
else:
    print("Figure display is disabled (SHOW_ALL_FIGURES = False).")


## Subset views

Sometimes you only want to look at the CSV inventory or only the figure
inventory. These two cells give you that without re-scanning.


In [ ]:
csv_inv = (
    inventory[inventory["suffix"].eq(".csv")]
    .sort_values(["parent", "name"])
    .reset_index(drop=True)
)
display_dataframe(csv_inv, title="\nCSV inventory\n")


In [ ]:
img_inv = (
    inventory[inventory["suffix"].isin([".png", ".jpg", ".jpeg", ".webp"])]
    .sort_values(["parent", "name"])
    .reset_index(drop=True)
)
display_dataframe(img_inv, title="\nFigure inventory\n")


## Optional: single-file preview

Set `SINGLE_CSV_TO_PREVIEW` or `SINGLE_IMAGE_TO_PREVIEW` in the parameters
cell to inspect one file in isolation. Leaving both `None` is the normal
case and these cells become no-ops.


In [ ]:
if SINGLE_CSV_TO_PREVIEW:
    p = Path(SINGLE_CSV_TO_PREVIEW)
    print(f"Previewing single CSV: {p}")
    display_dataframe(preview_csv_table(p, nrows=CSV_PREVIEW_ROWS))
else:
    print("(SINGLE_CSV_TO_PREVIEW is None)")


In [ ]:
if SINGLE_IMAGE_TO_PREVIEW:
    p = Path(SINGLE_IMAGE_TO_PREVIEW)
    print(f"Previewing single figure: {p}")
    show_image(p, title=p.name)
else:
    print("(SINGLE_IMAGE_TO_PREVIEW is None)")
